In [ ]:
!pip install -q requests pandas pandas_market_calendars datasets transformers \
               spacy beautifulsoup4 tqdm nltk pysentiment2
!python -m spacy download en_core_web_sm -q
import nltk; nltk.download('vader_lexicon')

In [ ]:
import torch
import os, re, time, datetime as dt, collections, requests, json, textwrap
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup
from transformers import pipeline
from tqdm.notebook import tqdm  # Colab‑friendly progress bar
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer


TICKERS  = ["AMZN","AMGN","AXP","BA","CAT","CRM","CSCO","CVX","DIS","DOW","GS",
            "HD","HON","IBM","INTC","JNJ","JPM","KO","MCD","MMM","MRK","MSFT",
            "NKE","PG","TRV","UNH","V","VZ","WBA","WMT"]
START    = "2024-03-31"        # inclusive lower bound
END      = "2025-03-31"        # inclusive upper bound
MAX_NEWS = 100                 # google max per query
OUT_DIR  = Path("/content")
UA_HDR   = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) "
                           "Chrome/122.0.0.0 Safari/537.36")}
LABEL2NUM = {"positive": 1.0, "neutral": 0.5, "negative": 0.0}



In [ ]:
# load models
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

device = 0 if torch.cuda.is_available() else -1
print(f"[INFO] running sentiment pipeline on device {device}")

print("[INFO] loading FinBERT")
FINBERT    = "ProsusAI/finbert"
tokenizer  = AutoTokenizer.from_pretrained(FINBERT)
fb_model   = AutoModelForSequenceClassification.from_pretrained(FINBERT)
finbert    = pipeline(
    "sentiment-analysis",
    model=fb_model,
    tokenizer=tokenizer,
    device=device,
    return_all_scores=True,
    top_k=None
)


In [ ]:
def google_html(query:str, n:int=100)->list[dict]:
    url = ("https://www.google.com/search"
           f"?q={requests.utils.quote(query)}&gl=us&tbm=nws&num={n}")
    html = requests.get(url, headers=UA_HDR, timeout=10).text
    soup = BeautifulSoup(html, "html.parser")
    hits=[]
    for el in soup.select("div.SoaBEf"):
        hits.append(dict(
            link    = el.find("a")["href"],
            title   = el.select_one("div.MBeuO").get_text(strip=True),
            snippet = el.select_one(".GI74Re").get_text(" ", strip=True),
            date_lab= el.select_one(".LfVVr").get_text(strip=True),
            source  = el.select_one(".NUnG9d span").get_text(strip=True)
        ))
    return hits

def label_to_date(label:str)->dt.date|None:
    label=label.lower(); today=dt.date.today()
    if "hour" in label or "min" in label: return today
    if "day"  in label:
        n=int(re.search(r"\d+", label)[0]); return today-dt.timedelta(days=n)
    try: return dt.datetime.strptime(label[:12], "%b %d, %Y").date()
    except: return None

def scrape_article(url:str)->str:
    try:
        html=requests.get(url, headers=UA_HDR, timeout=10).text
        soup=BeautifulSoup(html,"html.parser")
        return " ".join(p.get_text(' ',strip=True) for p in soup.find_all("p"))
    except: return ""


def text_cleaning(text: str) -> str:
    """Strip HTML, bracketed notes, and non‑alphanumerics except comma/apostrophe."""
    soup = BeautifulSoup(text, "html.parser")
    txt  = re.sub(r'\[[^]]*\]', '', soup.get_text())
    return re.sub(r"[^a-zA-Z0-9\s,']", "", txt)


In [ ]:
from datasets import Dataset
import xml.etree.ElementTree as ET
import datetime as dt
from tqdm.notebook import tqdm
import pandas as pd
import pandas_market_calendars as mcal
import time
import pysentiment2 as ps
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from transformers import pipeline
from nltk.sentiment.vader import SentimentIntensityAnalyzer


#  helper to fetch exactly one day's RSS headlines 
def fetch_rss_for_date(ticker: str, date: dt.date):
    after  = date.isoformat()
    before = (date + dt.timedelta(days=1)).isoformat()
    url    = (
      "https://news.google.com/rss/search"
      f"?q={requests.utils.quote(f'{ticker}+after:{after}+before:{before}')}"
      "&hl=en-US&gl=US&ceid=US:en"
    )
    r = requests.get(url, headers=UA_HDR, timeout=10)
    if r.status_code != 200:
        return []
    root = ET.fromstring(r.content)
    return [{
        "ticker": ticker,
        "date": after,
        "text": it.findtext("title","")
    } for it in root.findall(".//item")]

#  load sentiment models 
print("[INFO] loading VADER")
vader = SentimentIntensityAnalyzer()
print("[INFO] loading Loughran–McDonald")
lm = ps.LM()

#  NYSE trading‑day calendar
nyse         = mcal.get_calendar("NYSE")
schedule     = nyse.schedule(start_date=START, end_date=END)
trading_days = schedule.index.normalize().date                 # list of datetime.date
calendar_str = [d.isoformat() for d in trading_days]           # list of str

#  FinBERT pipeline on GPU 
finbert = pipeline(
    "sentiment-analysis",
    model="yiyanghkust/finbert-tone",
    device=0,
    batch_size=64,
    truncation=True,
    padding=True,
    max_length=512,
)

#  majority‑vote helper 
def vote_label(fb_score, vd_score, lm_score):
    votes = []
    votes.append("positive" if fb_score > 0.5 else "negative")
    if   vd_score > 0.6: votes.append("positive")
    elif vd_score < 0.4: votes.append("negative")
    else:               votes.append("neutral")
    if   lm_score > 0:  votes.append("positive")
    elif lm_score < 0:  votes.append("negative")
    else:               votes.append("neutral")
    majority = max(set(votes), key=votes.count)
    return LABEL2NUM[majority]

#  FETCH ALL (ticker, day) IN PARALLEL 
all_records = []
with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {
        executor.submit(fetch_rss_for_date, tic, day): (tic, day)
        for tic in TICKERS
        for day in trading_days
    }
    for fut in tqdm(as_completed(futures),
                    total=len(futures),
                    desc="Fetching all records"):
        records = fut.result()
        all_records.extend(records)
        time.sleep(0.1)   # throttle to avoid hammering RSS endpoint

#  BUILD ONE BIG DATASET & MAP VIA GPU 
ds = Dataset.from_list(all_records)

def map_signals(batch):
    texts     = [text_cleaning(t) for t in batch["text"]]
    fb_preds  = finbert(texts)
    vd_scores = [(vader.polarity_scores(t)["compound"] + 1) / 2 for t in texts]

    lm_scores = []
    for t in texts:
        cts = lm.get_score(lm.tokenize(t))
        pos = cts.get("Positive", 0); neg = cts.get("Negative", 0)
        lm_scores.append((pos - neg) / (pos + neg + 1))

    fb_vals = []
    for p in fb_preds:
        items = p if isinstance(p, list) else [p]
        pos   = next((d["score"] for d in items if d["label"].lower()=="positive"), None)
        neg   = next((d["score"] for d in items if d["label"].lower()=="negative"), None)
        fb_vals.append(((pos - neg + 1) / 2) if (pos and neg) else (pos or 0.5))

    return {"fb": fb_vals, "vd": vd_scores, "lm": lm_scores}

ds = ds.map(map_signals, batched=True, batch_size=64, remove_columns=["text"])

# TO PANDAS → AGGREGATE PER‑TICKER/DAY → FILL GAPS → WRITE CSVs 
df = ds.to_pandas()
combined = []

for tic, df_t in tqdm(df.groupby("ticker"), desc="Aggregating & writing"):
    daily = (
        df_t
        .groupby("date")[["fb","vd","lm"]]
        .mean()
        .reset_index()
        .assign(sentiment=lambda d: d.apply(lambda r: vote_label(r.fb, r.vd, r.lm), axis=1))
        .set_index("date")
        .reindex(calendar_str)
        .fillna({"sentiment": 0.5})
    )
    daily["ticker"] = tic
    out = OUT_DIR / f"{tic}_daily_sentiment.csv"
    daily.reset_index().to_csv(out, index=False)
    print(f"[INFO] {tic}: {len(daily)} days → {out.name}")
    combined.append(daily.reset_index())

if combined:
    all_df   = pd.concat(combined, ignore_index=True)
    all_path = OUT_DIR / "all_tickers_daily_sentiment.csv"
    all_df.to_csv(all_path, index=False)
    print(f"[INFO] combined CSV saved → {all_path.name}")
else:
    print("[WARN] no data generated")
